In [7]:
import sys; sys.path.insert(0, ".")
import pandas as pd
from src.data.db import get_connection
from src.processing.xg_processing import load_shots

conn = get_connection()
print(pd.read_sql("SELECT DISTINCT event_type FROM shot_events", conn))

df = load_shots(conn)
print(len(df))   # expect 566,545

     event_type
0  shot-on-goal
1  blocked-shot
2   missed-shot
3          goal
566545


In [8]:
from src.processing.xg_processing import load_shots, normalize_coordinates

df = load_shots(conn)
print((df.x_coord > 25).mean())    # before: about 0.50
df = normalize_coordinates(df)
print((df.x_coord > 25).mean())    # after: about 0.99

low = df[df.x_coord <= 25]

# 1. What zone does the API say these shots came from?
print(low.zone_code.value_counts())

# 2. Offensive-zone shots that ended up past the blue line, should be ~1.0
o = df[df.zone_code == "O"]
print((o.x_coord > 25).mean())

# 3. Where do the low shots sit?
print(pd.cut(low.x_coord, [-100, -25, 0, 25]).value_counts())
print(df.zone_code.value_counts(normalize=True))

o = df[df.zone_code == "O"]
n = df[df.zone_code == "N"]
print(o.x_coord.quantile(0.01))   # the shallowest offensive-zone shots
print(n.x_coord.max())            # the deepest neutral-zone shots

0.48421572867115587
0.9575550044568393
zone_code
N    14482
D     9069
O      496
Name: count, dtype: int64
0.9990864939231586
x_coord
(0, 25]        10745
(-100, -25]     9106
(-25, 0]        4196
Name: count, dtype: int64
zone_code
O    0.958376
N    0.025562
D    0.016062
Name: proportion, dtype: float64
28.0
24.0
